# Agentic Coding Workflow — Transparency Indicators

This notebook implements a single agentic workflow that replicates the human
coding scheme from Hardwicke et al. (2024).

**Steps:**
1. Check which articles the agent can access in full text on its own (open access).
2. Split articles into "accessible" vs. "needs a manually uploaded PDF" based on
   step 1's result.
3. For paywalled articles → you upload the PDF, the text is extracted, and the
   agent codes from that text — but it still has to actively verify external
   data/materials/analysis-script links, since those are often hosted separately
   from the article itself (e.g. on OSF, GitHub, ClinicalTrials.gov).
4. For accessible articles → the agent finds the full text itself and proceeds
   to full coding directly. If it turns out mid-coding that it actually cannot
   reach the text (even though step 1 said it could), the notebook pauses and
   asks you to upload a PDF for that article on the spot.
5. In all cases, the agent must actively open/verify links for every
   `*_access_status_agreed` field — it cannot just infer availability from the
   statement text alone. "Available upon request" always counts as not
   accessible.

**Reliability safeguards built into the coding step:**
- The model must return a strict, flat JSON structure (no nested fields).
- It must never claim it lacks internet access — it must actually call the
  web_search tool. If it never does, the attempt is automatically retried.
- It must never invent plausible-looking but unverified content (e.g.
  placeholder-style IDs or grant names) — every quote, ID, or number must come
  from something it actually read.
- Every `*_access_status_agreed` field includes the exact URL the model checked
  (`url_checked`), and the real `web_search` call history is logged separately
  (`_search_log`) as an independent audit trail, so results can be manually
  spot-checked for fabrication.

Results are saved to `agentic_coding_results_wide.csv` (one row per article, one
column per field, directly comparable to the human-coded ground truth) and
`agentic_search_log.csv` (the raw search audit trail).


In [ ]:
!pip install -q openai pandas pdfplumber

In [ ]:
from google.colab import userdata
from openai import OpenAI
import json
import time
import pandas as pd

secret_name = 'OPENAI_API_KEY2' # your API key
api_key = userdata.get(secret_name)

if not api_key:
    print(f"❌ ERROR: OpenAI key '{secret_name}' not found.")
else:
    client = OpenAI(api_key=api_key)
    print("✅ OpenAI client initialized.")

MODEL = "gpt-5.6-luna"

✅ OpenAI client initialized.


## Step 0 — Upload the article list

Upload a CSV file containing the articles to code. It should include at least
`article_id`, `DOI`, `journal_name`, and `title`.
The filename doesn't matter — whichever file you upload will be loaded
automatically.

In [ ]:
# Step 0: Upload the article list (CSV with columns like article_id, DOI, journal_name, title)
from google.colab import files

uploaded = files.upload()

# Read whichever file was actually uploaded, instead of hardcoding a filename
# (avoids errors if the uploaded file has a different name than expected)
uploaded_filename = list(uploaded.keys())[0]
pilot = pd.read_csv(uploaded_filename)

print(f"Loaded {len(pilot)} articles from '{uploaded_filename}'")
pilot


Saving articles_hardwicke_351_400.csv to articles_hardwicke_351_400.csv
Loaded 50 articles from 'articles_hardwicke_351_400.csv'


,article_id,DOI,journal_name,title,WOS,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 16262,Unnamed: 16263,Unnamed: 16264,Unnamed: 16265,Unnamed: 16266,Unnamed: 16267,Unnamed: 16268,Unnamed: 16269,Unnamed: 16270,Unnamed: 16271
0,47316,https://doi.org/10.1177/02724316211058065,JOURNAL OF EARLY ADOLESCENCE,Are Gifted Students More Victimized than Nongi...,WOS:000769286800001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,48249,https://doi.org/10.1080/23311908.2022.2047300,COGENT PSYCHOLOGY,Relationships between basic psychological need...,WOS:000766890300001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,74531,https://doi.org/10.1177/0894845320941251,JOURNAL OF CAREER DEVELOPMENT,Career Counselors Self-Disclosing to First-Gen...,WOS:000549934500001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,9481,https://doi.org/10.1002/pmh.1571,PERSONALITY AND MENTAL HEALTH,Late manifestation of borderline personality d...,WOS:000883845400001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,63307,https://doi.org/10.1016/j.apergo.2021.103600,APPLIED ERGONOMICS,"Activity space, office space: Measuring the sp...",WOS:000708130200008,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,55805,https://doi.org/10.1002/ijop.12827,INTERNATIONAL JOURNAL OF PSYCHOLOGY,The influence of ambient aroma on the middle s...,WOS:000744859500001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,55544,https://doi.org/10.1016/j.cognition.2021.104978,COGNITION,The beep-speed illusion: Non-spatial tones inc...,WOS:000745680700001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
7,48605,https://doi.org/10.1037/ppm0000383,PSYCHOLOGY OF POPULAR MEDIA,The Role of Different Characters on Story-Cons...,WOS:000766314500001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,74284,https://doi.org/10.1080/03004430.2020.1802260,EARLY CHILD DEVELOPMENT AND CARE,The effect of visual illusion on performance a...,WOS:000557987300001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,45645,https://doi.org/10.1080/1068316X.2022.2059480,PSYCHOLOGY CRIME & LAW,Assessing the deterrent effect of symbolic gua...,WOS:000778031300001,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Step 1 — Accessibility check

For each article, ask the model to try to find a freely accessible full-text
version (via DOI / title search). This does NOT yet do any coding — it only
determines whether the agent can proceed on its own (→ `accessible_ids`) or
will need a manually uploaded PDF (→ `paywalled_ids`) in the next step.

In [ ]:
# Step 1: Accessibility check
# For each article, ask the model to try to find a freely accessible full-text
# version (via DOI / title search). This step does NOT do any coding yet —
# it only determines whether the article can be found on its own, or will
# later need a manually uploaded PDF (see Step 3).

def check_accessibility(article):
    prompt = f"""Try to find a freely accessible, full-text version of the following
article — not just an abstract, and not a paywalled preview.

Title: {article['title']}
DOI: {article['DOI']}
Journal: {article['journal_name']}

Search the web (e.g. by DOI, by title, on the journal website, on preprint servers
such as PsyArXiv, or on institutional repositories).

Respond ONLY in JSON, no other text:
{{
  "article_id": "{article['article_id']}",
  "accessible": "YES" or "NO",
  "url_found": "<the URL of the full text you found, or empty string>",
  "note": "<one short sentence, e.g. what you tried>"
}}
"""
    response = client.responses.create(
        model=MODEL,
        tools=[{"type": "web_search"}],
        input=prompt,
    )

    # Strip markdown code fences the model sometimes adds around the JSON
    text = response.output_text.strip().replace("```json", "").replace("```", "").strip()

    try:
        return json.loads(text)
    except json.JSONDecodeError:
        # If parsing fails, default to "NO" (safer than assuming access)
        # and keep a snippet of the raw output for debugging
        return {
            "article_id": article["article_id"],
            "accessible": "NO",
            "url_found": "",
            "note": "parse_failed: " + text[:200],
        }


# Run the accessibility check on every article in the pilot sample
accessibility_results = []
for _, row in pilot.iterrows():
    article = row.to_dict()
    print(f"Checking access for article_id={article['article_id']} ({article['title'][:60]}...)")
    result = check_accessibility(article)
    accessibility_results.append(result)
    time.sleep(1)  # small delay to avoid hitting rate limits

access_df = pd.DataFrame(accessibility_results)
access_df.to_csv("accessibility_check.csv", index=False)
access_df

Checking access for article_id=47316 (Are Gifted Students More Victimized than Nongifted Students?...)
Checking access for article_id=48249 (Relationships between basic psychological needs and violent ...)
Checking access for article_id=74531 (Career Counselors Self-Disclosing to First-Generation Colleg...)
Checking access for article_id=9481 (Late manifestation of borderline personality disorder: Chara...)
Checking access for article_id=63307 (Activity space, office space: Measuring the spatial movement...)
Checking access for article_id=55805 (The influence of ambient aroma on the middle school students...)
Checking access for article_id=55544 (The beep-speed illusion: Non-spatial tones increase perceive...)
Checking access for article_id=48605 (The Role of Different Characters on Story-Consistent Attitud...)
Checking access for article_id=74284 (The effect of visual illusion on performance and quiet eye i...)
Checking access for article_id=45645 (Assessing the deterrent effect of sy

,article_id,accessible,url_found,note
0,47316,NO,,"I checked the DOI and journal page, ResearchGa..."
1,48249,YES,https://www.tandfonline.com/doi/full/10.1080/2...,I searched by DOI and title and found the free...
2,74531,NO,,"I searched the DOI, SAGE, ERIC, ResearchGate, ..."
3,9481,YES,https://onlinelibrary.wiley.com/doi/full/10.10...,I searched the DOI and title and found the pub...
4,63307,YES,https://www.sciencedirect.com/science/article/...,The ScienceDirect full-text article page is fr...
5,55805,YES,https://onlinelibrary.wiley.com/doi/10.1002/ij...,"I searched the DOI, title, Wiley Online Librar..."
6,55544,YES,https://psyarxiv.com/7cwm3/,I found a freely accessible author preprint vi...
7,48605,NO,,"I searched the DOI, exact title, journal recor..."
8,74284,YES,https://www.academia.edu/113379260/The_effect_...,Found a freely accessible full-text copy on Ac...
9,45645,YES,https://pure.port.ac.uk/ws/portalfiles/portal/...,Found a freely accessible full-text PDF in the...


## Step 2 — Split into accessible vs. paywalled

Splits articles into two groups based on Step 1's result: `accessible_ids`
(the agent found the text on its own) and `paywalled_ids` (will need a
manually uploaded PDF in Step 3).

As an optional sanity check, this also compares the agent's judgment against
the human-coded `open_access_status_agreed` column, to catch cases where the
agent's accessibility assessment disagrees with the original human coding.

In [ ]:
# Step 2: Split articles into two groups based on Step 1's results
# - accessible_ids: articles the model found on its own (will be coded directly)
# - paywalled_ids: articles that need a manually uploaded PDF (see Step 3)

accessible_ids = access_df[access_df["accessible"] == "YES"]["article_id"].astype(str).tolist()
paywalled_ids = access_df[access_df["accessible"] == "NO"]["article_id"].astype(str).tolist()

print(f"Accessible on its own: {len(accessible_ids)} articles -> {accessible_ids}")
print(f"Needs manual PDF upload: {len(paywalled_ids)} articles -> {paywalled_ids}")

# Optional sanity check: compare the model's accessibility judgment against the
# human-coded open_access_status_agreed column, to catch systematic disagreements
# early (e.g. the model wrongly assuming a paywall when the article is actually
# freely accessible).
if "open_access_status_agreed" in pilot.columns:
    pilot["article_id_str"] = pilot["article_id"].astype(str)
    check = pilot[["article_id_str", "open_access_status_agreed"]].merge(
        access_df.assign(article_id_str=access_df["article_id"].astype(str)),
        on="article_id_str", how="left"
    )
    print("\nComparison with human open_access label:")
    print(check[["article_id_str", "open_access_status_agreed", "accessible"]])

# Export a preliminary access-source CSV, based on Step 1's classification only.
# NOTE: this is a preliminary snapshot — an article classified here as
# "self_located" may still fall back to a manually supplied PDF mid-coding
# (see Step 6), and a "needs_pdf" article may end up with no PDF uploaded at
# all. A final, authoritative version of this column is produced at Step 7,
# once the full pipeline has run.
all_input_ids = set(pilot["article_id"].astype(str))
access_source_rows = []
for aid in all_input_ids:
    if aid in accessible_ids:
        source = "self_located (Step 1)"
    elif aid in paywalled_ids:
        source = "needs_pdf (Step 1)"
    else:
        source = "not_classified"
    access_source_rows.append({"article_id": aid, "access_source_preliminary": source})

access_source_df = pd.DataFrame(access_source_rows)
access_source_df.to_csv("access_source_preliminary.csv", index=False)
print(f"\nSaved preliminary access-source tracking for {len(access_source_df)} articles -> access_source_preliminary.csv")
access_source_df

Accessible on its own: 38 articles -> ['48249', '9481', '63307', '55805', '55544', '74284', '45645', '26145', '20987', '34647', '10727', '13251', '58259', '70087', '26521', '182', '21725', '1920', '33722', '68583', '73907', '72002', '120', '2826', '13937', '13823', '68570', '49234', '26570', '54655', '9690', '41743', '34919', '28347', '16531', '26478', '57888', '794']
Needs manual PDF upload: 12 articles -> ['47316', '74531', '48605', '37015', '12992', '2135', '65491', '71994', '33021', '5228', '55229', '47391']

Saved preliminary access-source tracking for 50 articles -> access_source_preliminary.csv


,article_id,access_source_preliminary
0,21725,self_located (Step 1)
1,16531,self_located (Step 1)
2,182,self_located (Step 1)
3,33722,self_located (Step 1)
4,26521,self_located (Step 1)
5,5228,needs_pdf (Step 1)
6,54655,self_located (Step 1)
7,13937,self_located (Step 1)
8,68570,self_located (Step 1)
9,55229,needs_pdf (Step 1)


## Step 3 — Upload PDFs for paywalled articles

For each article_id listed above as "needs manual PDF upload", place a PDF file
named exactly `<article_id>.pdf` (e.g. `53589.pdf`) and upload them all here.
You can select multiple files at once in the upload dialog (Ctrl/Cmd + click).

If there are no paywalled articles in this run, just skip this cell.

Note: if an article turns out to need a PDF later during coding (Step 6), even
though it wasn't listed here, the notebook will pause and ask you to upload it
at that point instead.

In [ ]:
# Step 3: Upload PDFs for paywalled articles
# For each article_id listed in paywalled_ids (from Step 2), upload a PDF file
# named exactly "<article_id>.pdf" (e.g. "53589.pdf"). You can select multiple
# files at once in the upload dialog (Ctrl/Cmd + click).

import pdfplumber

uploaded_pdfs = files.upload()

# Extract text from each uploaded PDF and store it in a dictionary keyed by
# article_id, so it can be looked up later during coding (Step 5/6)
pdf_texts = {}
for filename in uploaded_pdfs.keys():
    article_id = filename.replace(".pdf", "").strip()
    with pdfplumber.open(filename) as pdf:
        text = "\n".join(page.extract_text() or "" for page in pdf.pages)
    pdf_texts[article_id] = text
    print(f"Extracted {len(text)} characters from {filename} (article_id={article_id})")

print(f"\nTotal PDFs loaded: {len(pdf_texts)} -> {list(pdf_texts.keys())}")

Saving 47316.pdf to 47316.pdf
Saving 74531.pdf to 74531.pdf
Saving 37015.pdf to 37015.pdf
Saving 12992.pdf to 12992.pdf
Saving 2135.pdf to 2135.pdf
Saving 65491.pdf to 65491.pdf
Saving 71994.pdf to 71994.pdf
Saving 33021.pdf to 33021.pdf
Saving 5228.pdf to 5228.pdf
Saving 55229.pdf to 55229.pdf
Saving 47391.pdf to 47391.pdf
Extracted 49499 characters from 47316.pdf (article_id=47316)
Extracted 70197 characters from 74531.pdf (article_id=74531)
Extracted 84023 characters from 37015.pdf (article_id=37015)
Extracted 87634 characters from 12992.pdf (article_id=12992)
Extracted 38095 characters from 2135.pdf (article_id=2135)
Extracted 45178 characters from 65491.pdf (article_id=65491)
Extracted 42039 characters from 71994.pdf (article_id=71994)
Extracted 71562 characters from 33021.pdf (article_id=33021)
Extracted 58987 characters from 5228.pdf (article_id=5228)
Extracted 55368 characters from 55229.pdf (article_id=55229)
Extracted 49516 characters from 47391.pdf (article_id=47391)

Total 

## Step 4 — Full guideline definitions

Operational definitions for all coded fields, based on Hardwicke et al. (2024)
Method/Results and the finalized variable specification.

In [ ]:
# Step 4: Full guideline definitions
# Operational definitions for every field the model must code, based on
# Hardwicke et al. (2024) Method/Results and the finalized variable
# specification. This text is inserted directly into the coding prompt.

GUIDELINE_TEXT = """
You are coding a scientific article for research transparency indicators.

GENERAL RULES:
- Evaluate every field below. If a field is conditional (marked "only if..."), and the
  condition is not met, set its value to "NA".
- For every *_access_status field, you must ACTIVELY find and open the relevant link
  (e.g. a data repository, materials repository, preregistration, or analysis script
  repository) and verify whether it can really be accessed, downloaded, and viewed —
  do NOT infer this just from the statement text. "Available upon request" always
  counts as NO for access_status.
- Do not default to "NO"/"broken link"/"unclear access instructions" out of
  excessive caution. If a link resolves and shows the expected repository/content
  (even if the page layout is unfamiliar, or you cannot fully browse every file),
  that counts as accessible ("YES"). Only mark "NO" if you genuinely could not load
  the page at all, the page shows an explicit error (404, "page not found"), or the
  content is clearly behind a login/paywall/access request form.
- *_verbatim fields must be an EXACT quote from the article, not a paraphrase.
- If you cannot verify something (e.g. broken link, requires login), say so explicitly
  in the relevant *_access_detail or reason field instead of guessing.
- Read the ENTIRE article before concluding a statement is absent. Funding and
  conflict-of-interest statements are usually near the END of the article (after the
  Discussion, in Acknowledgments/Declarations), not in the abstract. Do not conclude
  "NO there is no funding statement" or "NO there is no conflict of interest
  statement" unless you have actually reached and read that part of the article.
- You MUST always respond using the exact JSON structure requested — never respond
  with plain prose, an apology, or a request for more information, even if you were
  unable to access the article. If you cannot access the full text at all, still
  return the JSON: set open_access_status_agreed to the appropriate value based on
  what you could determine, and set every other field's "value" to "NA" with a short
  note in the "verbatim" or "reason" field explaining you could not access the text.
- NEVER invent, guess, or generate placeholder-style content that merely looks
  plausible (e.g. example-style IDs such as "NCT12345678", generic names like
  "XYZ Grant 123" or "Grant 123", or any invented registry number, DOI, author
  name, or figure). Every verbatim quote, ID, URL, and number you report MUST be
  something you actually read in the real article or web page — never something
  you inferred should probably be there based on the pattern of similar articles.
  If you did not actually see a specific ID, number, or exact quote, leave that
  field as "NA" or write "not verified" rather than filling in a plausible-looking
  value.

FIELDS TO CODE:

0. open_access_status_agreed: whether a free, full-text version of the article is
   available. Values: "Yes" / "No I can only access the article through a paywall"

   IMPORTANT: Do not conclude "paywall" after a single search or after seeing a
   publisher page that shows only an abstract. Many articles have a free version
   elsewhere. Before answering "No", try AT LEAST these strategies:
   (a) search by the DOI directly, (b) search by the exact title as a general web
   search, (c) search for the title plus "PDF", "PMC", "preprint", "OSF", or
   "free full text", (d) check PubMed Central (PMC) and preprint servers (e.g.
   PsyArXiv). Only answer "No I can only access the article through a paywall" if
   none of these searches surface a free full-text version.
   If a full-text URL was already found in a prior step (given to you below as
   "KNOWN ACCESSIBLE URL"), use that URL as your starting point — actually open it
   and read the full article there, do not search from scratch or give up early.

1. prereg_statement: whether the article states the study (or an aspect of it) was
   preregistered BEFORE data collection/analysis, in a registry (e.g. OSF, AsPredicted,
   ClinicalTrials.gov, PROSPERO).

   IMPORTANT: Preregistration statements and links (e.g. to AsPredicted, OSF Registries)
   can appear anywhere in the article — in the abstract, Methods section, or as a
   footnote — not only in a dedicated "Transparency" section. Scan the ENTIRE article
   text for registry names (OSF, AsPredicted, ClinicalTrials.gov, PROSPERO, etc.) or
   registration numbers/URLs before concluding there is no statement.

   Values: "NO there is no statement about preregistration" /
           "YES the statement says that there is a preregistration" /
           "YES the statement says that there is no preregistration"
   1a. prereg_statement_verbatim: exact quote
   1b. prereg_access_method: which registry (only if "there is a preregistration")
       Values: Open Science Framework / ClinicalTrials.gov / AsPredicted / PROSPERO /
               Netherlands Trial Register / Clinical Research Information Service /
               Chinese Clinical Trial Registry / Both Open Science Framework and
               AsPredicted / UNCLEAR
   1c. prereg_access_status: actually open the link. YES/NO (only if "there is a
       preregistration")
   1d. prereg_access_detail: if NO, why — "Broken link" / "Unclear access instructions"

2. data_statement: whether the article states RAW/PRIMARY data (individual-level, NOT
   summary/aggregated data) are available. If a third-party data source is cited
   WITHOUT a clear statement of availability, use the UNCLEAR category — do not count
   as available.
   Values: "NO there is no data availability statement" /
           "YES the statement says that the data are available" /
           "YES the statement says that the data are not available" /
           "UNCLEAR a third party source is identified without any indication of availability"
   2a. data_statement_verbatim: exact quote
   2b. data_reason: stated reason if "not available" —
       Not applicable / Not permitted by third party / Participant privacy /
       No participant consent / Data collection ongoing
   2c. data_access_method: how data are accessed (only if "available") —
       Upon request from the authors / An online, third-party repository (e.g., OSF,
       FigShare etc.) / From a third party / Personal or institution website /
       Supplementary materials hosted by the journal / UNCLEAR
   2d. data_access_status: ACTUALLY open the link and try to download. YES/NO (only if
       "available"). "Upon request" = NO.
   2e. data_access_detail: if NO — "Broken link" / "Unclear access instructions" /
       "Upon request"

3. materials_statement: same logic as data, but for research materials (e.g. survey
   instruments, stimuli, non-analysis code).
   Values: "NO there is no materials availability statement" /
           "YES the statement says that the materials are available" /
           "YES the statement says that the materials are not available" /
           "UNCLEAR a third party source is identified without any indication of availability"

   IMPORTANT — distinguish two very different situations when the article names a
   measure/instrument:

   (a) STANDARDIZED, PUBLISHED SCALES/QUESTIONNAIRES cited with an original
   publication reference (e.g. "the 6-item Gratitude Questionnaire (McCullough
   et al., 2002)", "the Mental Health Continuum-Short Form (Keyes, 2013)", "the
   Reading the Mind in the Eyes Test"). These are established, publicly known
   instruments that any researcher could obtain independently by looking up the
   cited publication — mentioning them is NOT a materials availability concern
   in this coding scheme. Use "NO there is no materials availability statement"
   unless the article ALSO makes an explicit statement about sharing/accessing
   ITS OWN version of the materials (e.g. modified items, translated version,
   full item wording as an appendix).

   (b) STUDY-SPECIFIC OR THIRD-PARTY-OWNED MATERIALS without a clear availability
statement — e.g. custom-made stimuli, video clips, a proprietary/unpublished
instrument, or materials attributed to "the project team" or an unnamed
internal source... This also includes generic, unnamed references to specific
measurement tools (e.g., "two psychometric measures were used," "a
questionnaire assessing X was administered") where the article does not
name the instrument or cite its source — the mere fact that a measure was
used, without identifying it, is enough to trigger UNCLEAR.

    In short: a bare citation to a well-known, previously published scale = NO
   statement. A reference to unpublished, custom, or ambiguously-sourced
   material with no access information = UNCLEAR.

   3a. materials_statement_verbatim: exact quote
   3b. materials_reason: stated reason if "not available" —
       Data collection ongoing / No participant consent / Not applicable /
       Not intended for public use / Not permitted by third party
   3c. materials_access_method: (only if "available") —
       An online, third-party repository (e.g., OSF, FigShare etc.) / Supplementary
       materials hosted by the journal / Within the article (or appendices) / Upon
       request from the authors / Personal or institution website / From a third
       party / UNCLEAR
   3d. materials_access_status: ACTUALLY open the link and try to download. YES/NO
       (only if "available"). "Upon request" = NO.
   3e. materials_access_detail: if NO — "Broken link" / "Unclear access instructions" /
       "Upon request"

4. analysis_statement: whether analysis scripts/code (or point-and-click instructions)
   used to produce the reported results are available. (No UNCLEAR category here.)
   Values: "NO there is no analysis script availability statement" /
           "YES the statement says that the analysis scripts are available" /
           "YES the statement says that the analysis scripts are not available"
   4a. analysis_statement_verbatim: exact quote
   4b. analysis_reason: stated reason if "not available" —
       Not applicable / Data collection ongoing
   4c. analysis_access_method: (only if "available") —
       An online, third-party repository (e.g., OSF, FigShare etc.) / Upon request
       from the authors / Supplementary materials hosted by the journal / Within the
       article (or appendices)
   4d. analysis_access_status: ACTUALLY open the link and try to download. YES/NO
       (only if "available"). "Upon request" = NO.
   4e. analysis_access_detail: if NO — "Broken link" / "Upon request"

5. funding_statement: whether the article includes an explicit funding statement
   (disclosing a source, OR explicitly stating there was none). Absence of any
   mention = "NO there is no funding statement". Use UNCLEAR only if a statement
   exists but is ambiguous.

   IMPORTANT: Funding statements are very often located AFTER the main text, near
   the end of the article — commonly under headings like "Funding", "Acknowledgments",
   "Declaration of Interest", "Financial Support", or in a "Transparency" section.
   If you searched but only read the abstract/introduction, you have NOT verified
   this field yet — you must locate and read the end matter of the article
   (Acknowledgments/Declarations/Funding section) before answering "NO there is no
   funding statement".

   Values: "NO there is no funding statement" /
           "YES the statement discloses at least one source of funding" /
           "YES the statement says that there was no relevant funding" / "UNCLEAR"
   5a. funding_statement_verbatim: exact quote

6. coi_statement: whether the article includes an explicit conflict-of-interest
   statement (disclosing a conflict, OR explicitly stating there were none). Absence
   of any mention = "NO there is no conflict of interest statement". Use UNCLEAR only
   if a statement exists but is ambiguous.
   Values: "NO there is no conflict of interest statement" /
           "YES the statement says that there were no conflicts of interest" /
           "YES the statement discloses at least one conflict of interest" / "UNCLEAR"
   6a. coi_statement_verbatim: exact quote

7. research_design_agreed: classify the overall research design of the study.
   This is a multi-select field — if the article combines both experimental and
   non-experimental components, use the combined value.
   Values: "Experimental" / "Non-experimental" / "Meta-analysis" /
           "Experimental, Non-experimental"
"""
print(f"Guideline text length: {len(GUIDELINE_TEXT)} characters")

Guideline text length: 12495 characters


## Step 5 — Coding function

Handles both branches: (a) accessible articles — agent finds the article
itself; (b) paywalled articles — full text is provided directly, but the
agent still has to actively verify external links for data/materials/
analysis/prereg access. Both functions retry automatically if the response
fails to parse. For self-located articles, the pipeline additionally retries
if the model never actually called `web_search` (a sign it may be answering
from guesswork instead of real verification); this additional safeguard is
NOT applied to paywalled/PDF-coded articles, which are accepted once the
response parses correctly, regardless of whether web_search was called.

In [ ]:
# Step 4 (continued): Response template + coding functions
# RESPONSE_TEMPLATE defines the exact, flat JSON structure the model must return.
# extract_search_log() pulls the real web_search call history from the API
# response, as an independent audit trail (separate from what the model claims).
# safe_parse_json() robustly extracts JSON even if the model adds stray text.
# code_accessible_article() / code_paywalled_article() run the actual coding call.

RESPONSE_TEMPLATE = """
Respond ONLY in JSON, no other text, no markdown fences, in exactly this structure.
Field names below are FINAL — use them exactly as written, they map directly onto
the human-coded dataset columns.

CRITICAL: All fields listed below (e.g. "data_statement_agreed", "data_reason",
"data_access_method_agreed", "data_access_status_agreed") are SEPARATE, SIBLING
top-level keys in the JSON object — do NOT nest any of them inside another field's
object, even if they relate to the same topic. Follow the exact flat structure
shown below, with every field at the same nesting level.

CRITICAL: You DO have real-time internet access via the web_search tool provided to
you. Never claim you lack internet access or cannot check external sources — you
must actually use the web_search tool to look things up. If, after genuinely trying,
you still cannot access something, reflect that by setting the relevant field's
"value" to "NA" (or the appropriate "not accessible" category) — do NOT respond with
plain prose explaining the limitation. Your entire response must always be the JSON
object below, with no exceptions.

For every *_access_status_agreed field, also include "url_checked": the exact URL
you opened to verify access (empty string if not applicable / NA).

{
  "article_id": "<id>",
  "open_access_status_agreed": {"value": "..."},

  "prereg_statement_agreed": {"value": "...", "verbatim": "..."},
  "prereg_access_method_agreed": {"value": "..."},
  "prereg_access_status_agreed": {"value": "...", "url_checked": "..."},
  "prereg_access_detail": {"value": "..."},

  "data_statement_agreed": {"value": "...", "verbatim": "..."},
  "data_reason": {"value": "..."},
  "data_access_method_agreed": {"value": "..."},
  "data_access_status_agreed": {"value": "...", "url_checked": "..."},
  "data_access_detail": {"value": "..."},

  "materials_statement_agreed": {"value": "...", "verbatim": "..."},
  "materials_reason": {"value": "..."},
  "materials_access_method_agreed": {"value": "..."},
  "materials_access_status_agreed": {"value": "...", "url_checked": "..."},
  "materials_access_detail": {"value": "..."},

  "analysis_statement_agreed": {"value": "...", "verbatim": "..."},
  "analysis_reason": {"value": "..."},
  "analysis_access_method_agreed": {"value": "..."},
  "analysis_access_status_agreed": {"value": "...", "url_checked": "..."},
  "analysis_access_detail": {"value": "..."},

  "funding_statement_agreed": {"value": "...", "verbatim": "..."},
  "coi_statement_agreed": {"value": "...", "verbatim": "..."},
  "research_design_agreed": {"value": "..."}
}
"""


def extract_search_log(response):
    """Pull out the actual web_search tool calls the model made — this is the
    real audit trail proving what it searched for / opened, independent of what
    it claims in url_checked."""
    log = []
    for item in response.output:
        if getattr(item, "type", None) == "web_search_call":
            action = getattr(item, "action", None)
            log.append({
                "status": getattr(item, "status", None),
                "query": getattr(action, "query", None) if action else None,
                "url": getattr(action, "url", None) if action else None,
            })
    return log


def safe_parse_json(raw_text, article_id):
    """Robustly extract a JSON object from the model's response, even if it added
    stray text (e.g. "JSON output:") before or after the actual JSON."""
    cleaned = raw_text.strip().replace("```json", "").replace("```", "").strip()
    start = cleaned.find("{")
    end = cleaned.rfind("}")
    if start == -1 or end == -1 or end < start:
        return {"article_id": article_id, "error": "parse_failed", "raw": raw_text}
    candidate = cleaned[start:end + 1]
    try:
        return json.loads(candidate)
    except json.JSONDecodeError:
        return {"article_id": article_id, "error": "parse_failed", "raw": raw_text}


def code_accessible_article(article, known_url=None, max_attempts=3):
    """Code an article the model was able to find on its own (Step 1 said
    'accessible'). Retries if the response fails to parse, or if the model
    never actually called web_search (a sign it may be fabricating instead of
    verifying)."""
    known_url_block = (
        f"KNOWN ACCESSIBLE URL: {known_url}\n"
        "This URL was already confirmed to lead to a free full-text version of this "
        "article in a prior step. You MUST still actually call the web_search tool "
        "to fetch and read the real content at or near this URL before answering — "
        "having this URL does NOT mean you can skip verification or answer from "
        "memory/inference. If you do not call web_search at all, your answer will "
        "be rejected.\n"
        if known_url else
        "No previously known URL is available for this article — you must find the "
        "full text yourself (web search, by DOI/title).\n"
    )
    result = None
    for attempt in range(max_attempts):
        prompt = f"""{GUIDELINE_TEXT}

ARTICLE TO CODE:
Title: {article['title']}
DOI: {article['DOI']}
Journal: {article['journal_name']}

{known_url_block}
Code all fields listed above based on the full text, actively opening any
relevant links for the *_access_status_agreed fields.

{RESPONSE_TEMPLATE.replace('<id>', str(article['article_id']))}
"""
        response = client.responses.create(
            model=MODEL,
            tools=[{"type": "web_search"}],
            input=prompt,
            max_output_tokens=4000,
        )
        result = safe_parse_json(response.output_text, article["article_id"])

        if "error" in result:
            print(f"    (attempt {attempt+1}/{max_attempts} failed to parse, retrying...)")
            continue

        search_log = extract_search_log(response)
        result["_search_log"] = search_log
        if len(search_log) == 0:
            print(f"    (attempt {attempt+1}/{max_attempts}: model never called web_search, retrying...)")
            continue

        return result

    # All attempts exhausted — return the last result, flagged as unverified
    result["_no_search_performed"] = True
    return result


def code_paywalled_article(article, full_text, max_attempts=2):
    """Code an article using manually provided full text (extracted from an
    uploaded PDF). The model still must actively verify any external
    data/materials/analysis-script links via web_search, since those are often
    hosted outside the paywalled article itself."""
    for attempt in range(max_attempts):
        prompt = f"""{GUIDELINE_TEXT}

ARTICLE TO CODE:
Title: {article['title']}
DOI: {article['DOI']}
Journal: {article['journal_name']}

The full text of the article is provided below (extracted from a PDF you cannot
access online yourself). Code the *_statement_agreed, *_verbatim, and *_reason
fields based on this text. HOWEVER, for the *_access_status_agreed fields, any
linked data/materials/analysis-script/preregistration repositories may be hosted
externally (e.g. on OSF, FigShare, ClinicalTrials.gov) — you MUST still actively
search for and open those external links to verify access, using the web search
tool.

--- FULL TEXT START ---
{full_text[:12000]}
{"[... middle of article omitted for length ...]" if len(full_text) > 16000 else ""}
{full_text[-4000:] if len(full_text) > 16000 else ""}
--- FULL TEXT END ---

{RESPONSE_TEMPLATE.replace('<id>', str(article['article_id']))}
"""
        response = client.responses.create(
            model=MODEL,
            tools=[{"type": "web_search"}],
            input=prompt,
            max_output_tokens=4000,
        )
        result = safe_parse_json(response.output_text, article["article_id"])
        if "error" not in result:
            result["_search_log"] = extract_search_log(response)
            return result
        print(f"    (attempt {attempt+1}/{max_attempts} failed, retrying...)")
    return result

## Step 6 — Run coding over all articles

Routes each article to the correct branch based on Step 2's classification.
If an article classified as "accessible" turns out to be unreachable once the
model actually tries to code it, this step pauses and asks you to upload a PDF
for that article right away, instead of silently producing an incomplete
result.

In [ ]:
# Step 6: Run coding over all articles
# Routes each article to the correct branch based on Step 2's classification:
#   - "accessible" articles -> model finds the text itself
#   - "paywalled" articles with an uploaded PDF -> coded from that PDF text
#   - "paywalled" articles without a PDF -> skipped (upload one first)
# If an "accessible" article turns out to be unreachable during coding (the
# model gives up mid-way, even though Step 1 said it was accessible), this
# loop pauses and asks for a PDF upload right there, instead of silently
# producing an incomplete result.

# Build a lookup so the coding step can reuse the URL already found in Step 1,
# instead of re-searching from scratch (and possibly giving up too early)
url_lookup = {
    str(row["article_id"]): row.get("url_found", "")
    for _, row in access_df.iterrows()
}

all_results = []
access_source_log = []  # NEW: tracks how each article's full text was obtained

for _, row in pilot.iterrows():
    article = row.to_dict()
    aid = str(article["article_id"])
    print(f"Coding article_id={aid} ({article['title'][:60]}...)")

    if aid in accessible_ids:
        result = code_accessible_article(article, known_url=url_lookup.get(aid))
        source = "self_located"  # NEW: default assumption for this branch

        if "error" not in result:
            oa_value = result.get("open_access_status_agreed", {}).get("value", "")
            if "paywall" in oa_value.lower() or "cannot" in oa_value.lower() or "could not" in str(result).lower():
                print(f"  ⚠️  Article was classified as 'accessible' in Step 1, but the model")
                print(f"      could not reach the full text during coding. Please upload a PDF")
                print(f"      for article_id={aid}.")
                uploaded_pdf = files.upload()
                if uploaded_pdf:  # NEW: check whether a file was actually provided
                    for filename in uploaded_pdf.keys():
                        with pdfplumber.open(filename) as pdf:
                            text = "\n".join(page.extract_text() or "" for page in pdf.pages)
                        pdf_texts[aid] = text
                        print(f"  Extracted {len(text)} characters from {filename}, re-coding...")
                    result = code_paywalled_article(article, pdf_texts[aid])
                    source = "self_located_then_pdf_fallback"  # NEW
                else:
                    print(f"  SKIPPED — no PDF supplied for article_id={aid} after access failure")
                    access_source_log.append({"article_id": aid, "access_source": "excluded_no_pdf_after_failure"})
                    continue

    elif aid in paywalled_ids and aid in pdf_texts:
        result = code_paywalled_article(article, pdf_texts[aid])
        source = "pdf_provided"  # NEW

    elif aid in paywalled_ids:
        print(f"  SKIPPED — no PDF uploaded for article_id={aid}")
        access_source_log.append({"article_id": aid, "access_source": "excluded_no_pdf_provided"})  # NEW
        continue
    else:
        print(f"  SKIPPED — article_id={aid} not classified in Step 1")
        access_source_log.append({"article_id": aid, "access_source": "excluded_not_classified"})  # NEW
        continue

    if "error" in result:
        print("  parse error:", result.get("raw", "")[:150])
    else:
        n_searches = len(result.get("_search_log", []))
        print(f"  OK — model made {n_searches} web_search call(s)")

    all_results.append(result)
    access_source_log.append({"article_id": aid, "access_source": source})  # NEW
    time.sleep(1)

print(f"\nCoded {len(all_results)} articles.")

Coding article_id=47316 (Are Gifted Students More Victimized than Nongifted Students?...)
  OK — model made 6 web_search call(s)
Coding article_id=48249 (Relationships between basic psychological needs and violent ...)
  OK — model made 7 web_search call(s)
Coding article_id=74531 (Career Counselors Self-Disclosing to First-Generation Colleg...)
  OK — model made 4 web_search call(s)
Coding article_id=9481 (Late manifestation of borderline personality disorder: Chara...)
  OK — model made 8 web_search call(s)
Coding article_id=63307 (Activity space, office space: Measuring the spatial movement...)
  OK — model made 6 web_search call(s)
Coding article_id=55805 (The influence of ambient aroma on the middle school students...)
  OK — model made 3 web_search call(s)
Coding article_id=55544 (The beep-speed illusion: Non-spatial tones increase perceive...)
  OK — model made 8 web_search call(s)
Coding article_id=48605 (The Role of Different Characters on Story-Consistent Attitud...)
  SKIPPE

Saving 2826.pdf to 2826.pdf
  Extracted 82 characters from 2826.pdf, re-coding...
  OK — model made 8 web_search call(s)
Coding article_id=13937 ("Sorry, I Didn't Mean to Kiss at You": A Descriptive Analysi...)
  OK — model made 11 web_search call(s)
Coding article_id=13823 (A test of indirect grounding of abstract concepts using mult...)
  OK — model made 6 web_search call(s)
Coding article_id=68570 (The malleability of collective memories: One year after the ...)
  OK — model made 9 web_search call(s)
Coding article_id=49234 (Physiological Foundations for Religious Experiences in Devot...)
  OK — model made 8 web_search call(s)
Coding article_id=26570 (Self-relevance and the activation of attentional networks...)
  OK — model made 7 web_search call(s)
Coding article_id=54655 (Smoking Behavior and Driver's Involvement in Road Traffic Ac...)
  OK — model made 3 web_search call(s)
Coding article_id=9690 (Longitudinal trajectories of coronavirus anxiety and health ...)
  OK — model made 

## Step 7 — Flatten results and convert to wide format

Converts each article's nested JSON result into a wide table (one row per
article, one column per field), matching the structure of the human-coded
dataset so the two can be compared directly. Saves the final results and the
raw `web_search` audit log to CSV.

In [ ]:
# Step 7: Flatten results and convert to wide format
# Each model response is a nested JSON object (one key per field). This step:
#   1. Flattens it into a "long" table (one row per article_id + field)
#   2. Pivots that into a "wide" table (one row per article, one column per
#      field) matching the structure of the human-coded dataset, so the two
#      can be compared directly
#   3. Saves both the final results and the raw web_search audit log to CSV

# Fields that carry a verbatim quote (statement fields only)
STATEMENT_FIELDS = [
    "prereg_statement_agreed", "data_statement_agreed",
    "materials_statement_agreed", "analysis_statement_agreed",
    "funding_statement_agreed", "coi_statement_agreed",
]

# Fields where the model had to actively open a link (carry a url_checked)
ACCESS_STATUS_FIELDS = [
    "prereg_access_status_agreed", "data_access_status_agreed",
    "materials_access_status_agreed", "analysis_access_status_agreed",
]

def flatten(result):
    """Convert one article's nested JSON result into several flat rows:
    one row per (article_id, field), with value / verbatim / url_checked
    pulled out as separate columns."""
    rows = []
    article_id = result.get("article_id")
    for field, content in result.items():
        if field in ("article_id", "_search_log") or not isinstance(content, dict):
            continue
        rows.append({
            "article_id": article_id,
            "field": field,
            "value": content.get("value", ""),
            "verbatim": content.get("verbatim", ""),
            "url_checked": content.get("url_checked", ""),
        })
    return rows

flat_rows = []
search_log_rows = []
for r in all_results:
    if "error" in r:
        continue
    flat_rows += flatten(r)
    for entry in r.get("_search_log", []):
        search_log_rows.append({"article_id": r.get("article_id"), **entry})

long_df = pd.DataFrame(flat_rows)

# Pivot the "value" column to wide format — one row per article. Field names
# already match the final column names, so no renaming is needed here.
value_wide = long_df.pivot(index="article_id", columns="field", values="value").reset_index()

# Pivot the "verbatim" column separately, only for statement fields, and
# rename columns from "..._agreed" to "..._verbatim"
verbatim_wide = (
    long_df[long_df["field"].isin(STATEMENT_FIELDS)]
    .pivot(index="article_id", columns="field", values="verbatim")
    .rename(columns=lambda c: c.replace("_agreed", "_verbatim"))
    .reset_index()
)

# Pivot the "url_checked" column separately, only for access_status fields, and
# rename columns from "..._status_agreed" to "..._url_checked"
url_checked_wide = (
    long_df[long_df["field"].isin(ACCESS_STATUS_FIELDS)]
    .pivot(index="article_id", columns="field", values="url_checked")
    .rename(columns=lambda c: c.replace("_status_agreed", "_url_checked"))
    .reset_index()
)

# Merge all three pieces (value, verbatim, url_checked) into one wide table
results_df = value_wide.merge(verbatim_wide, on="article_id", how="left").merge(
    url_checked_wide, on="article_id", how="left"
)

access_source_df = pd.DataFrame(access_source_log)
access_source_df["article_id"] = access_source_df["article_id"].astype(results_df["article_id"].dtype)
results_df = results_df.merge(access_source_df, on="article_id", how="left")

# Reorder columns to match the finalized variable specification, for easier
# side-by-side comparison with the human-coded dataset
ordered_cols = ["article_id", "access_source",
    "open_access_status_agreed",
    "prereg_statement_agreed", "prereg_statement_verbatim", "prereg_access_method_agreed",
    "prereg_access_status_agreed", "prereg_access_url_checked", "prereg_access_detail",
    "data_statement_agreed", "data_statement_verbatim", "data_reason", "data_access_method_agreed",
    "data_access_status_agreed", "data_access_url_checked", "data_access_detail",
    "materials_statement_agreed", "materials_statement_verbatim", "materials_reason", "materials_access_method_agreed",
    "materials_access_status_agreed", "materials_access_url_checked", "materials_access_detail",
    "analysis_statement_agreed", "analysis_statement_verbatim", "analysis_reason", "analysis_access_method_agreed",
    "analysis_access_status_agreed", "analysis_access_url_checked", "analysis_access_detail",
    "funding_statement_agreed", "funding_statement_verbatim",
    "coi_statement_agreed", "coi_statement_verbatim", "research_design_agreed",
]
results_df = results_df[[c for c in ordered_cols if c in results_df.columns]]

# Save final results and the raw search audit log
results_df.to_csv("agentic_coding_results_wide.csv", index=False)

search_log_df = pd.DataFrame(search_log_rows)
search_log_df.to_csv("agentic_search_log.csv", index=False)

results_df

,article_id,access_source,open_access_status_agreed,prereg_statement_agreed,prereg_statement_verbatim,prereg_access_method_agreed,prereg_access_status_agreed,prereg_access_url_checked,prereg_access_detail,data_statement_agreed,...,analysis_reason,analysis_access_method_agreed,analysis_access_status_agreed,analysis_access_url_checked,analysis_access_detail,funding_statement_agreed,funding_statement_verbatim,coi_statement_agreed,coi_statement_verbatim,research_design_agreed
0,10727,self_located,Yes,NO there is no statement about preregistration,NA,NA,NA,,NA,NO there is no data availability statement,...,Not applicable,NA,NA,,NA,YES the statement discloses at least one sourc...,"This research has been conducted as part of ""E...",YES the statement says that there were no conf...,None.,Experimental
1,120,self_located,Yes,NO there is no statement about preregistration,NA,NA,NA,,NA,UNCLEAR a third party source is identified wit...,...,Not applicable,NA,NA,,NA,YES the statement says that there was no relev...,FINANCIAL SUPPORT: None,YES the statement says that there were no conf...,CONFLICT OF INTEREST: None,Non-experimental
2,12992,pdf_provided,No I can only access the article through a pay...,NA,NA — the accessible article text was incomplet...,NA,NA,,NA,NA,...,NA,NA,NA,,NA,NA,NA — the accessible article text was incomplet...,NA,NA — the accessible article text was incomplet...,Non-experimental
3,13251,self_located,Yes,NO there is no statement about preregistration,No preregistration statement found in the full...,NA,NA,,NA,YES the statement says that the data are not a...,...,NA,NA,NA,,NA,YES the statement discloses at least one sourc...,This research was made possible by a grant to ...,YES the statement says that there were no conf...,No conflict of interest.,Non-experimental
4,13823,self_located,Yes,NO there is no statement about preregistration,NA,NA,NA,,NA,YES the statement says that the data are avail...,...,NA,NA,NA,,NA,YES the statement discloses at least one sourc...,This research was supported by JSPS KAKENHI Gr...,YES the statement says that there were no conf...,The author declares that the research was cond...,Experimental
5,13937,self_located,Yes,NO there is no statement about preregistration,NA,NA,NA,,NA,NO there is no data availability statement,...,Not applicable,NA,NA,,NA,YES the statement discloses at least one sourc...,This work was supported by the University of W...,YES the statement says that there were no conf...,The authors have no relevant financial or non-...,Non-experimental
6,16531,self_located,Yes,NO there is no statement about preregistration,NA,NA,NA,,NA,NO there is no data availability statement,...,Not applicable,NA,NA,,NA,NO there is no funding statement,NA,YES the statement says that there were no conf...,The authors declare no conflicts of interest.,Non-experimental
7,182,self_located,Yes,NO there is no statement about preregistration,NA,NA,NA,,NA,NO there is no data availability statement,...,Not applicable,NA,NA,,NA,NO there is no funding statement,NA,NO there is no conflict of interest statement,NA,Non-experimental
8,1920,self_located,Yes,NO there is no statement about preregistration,NA,NA,NA,,NA,NO there is no data availability statement,...,NA,NA,NA,,NA,YES the statement discloses at least one sourc...,Funding for this work was provided by the Bost...,YES the statement says that there were no conf...,None of the authors report a conflict of inter...,Non-experimental
9,20987,self_located,Yes,YES the statement says that there is a preregi...,The design and analysis plans for the experime...,Open Science Framework,NO,"https://osf.io/dt7vy, https://osf.io/pqhvn",Broken link,YES the statement says that the data are avail...,...,Not applicable,"An online, third-party repository (e.g., OSF, ...",NO,https://osf.io/r7bes/,Broken link,NO there is no funding statement,NA,YES the statement says that there were no conf...,The author(s) declared that there were no conf...,"Experimental, Non-experimental

In [ ]:
from google.colab import files
files.download("agentic_coding_results_wide.csv")
files.download("accessibility_check.csv")
files.download("agentic_search_log.csv")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>